# Holi qualitative example — Mistral-7B-Instruct (Appendix G)

Reproduces **Appendix G, "Item-Level Knockout Example"** of the paper: the
R→item edge knockout of the three identified cultural-binding heads
(L8H16, L9H23, L12H9), evaluated on the five factorial pairs covering the
Holi festival, at baseline and after knockout (published run: |ΔS| = 6.00 →
3.95, a 34.2% item-level reduction, vs. the 23.5% average across all 66
items in Table 2).

This example is Mistral-7B-Instruct only, so `MODEL_KEY` is fixed.

In [ ]:
MODEL_KEY = "mistral"  # fixed: the Appendix G example is Mistral-7B-Instruct only

In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)

from common import config

CFG = config.init(MODEL_KEY, "instruct")
SEED = config.SEED
DATA_DIR = config.DATA_DIR
HF_TOKEN = config.HF_TOKEN          # read from the HF_TOKEN env var
OUTPUT_DIR = config.OUTPUT_DIR

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from common.text_parsers import extract_options
from common.instruct.data import load_n4, build_factorial_as_conditions
from common.instruct.prompts import format_for_chat, find_option_token_ids, detect_spans
from common.instruct.hooks import edge_knockout, compute_logit_scores_edge

In [ ]:
# Load data + model
# Load data
cultural_items, neutral_items = load_n4(DATA_DIR)
data = build_factorial_as_conditions(cultural_items, seed=SEED)
n_total = len(data['B_cult'])
print(f"  {n_total} examples, {len(set(data['scenarios']))} scenarios")

# Load model
print(f"\n  Loading {CFG['model_path']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG['model_path'], trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_path'], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

option_tokens_raw = find_option_token_ids(tokenizer)
first_device = next(model.parameters()).device
option_tokens = {opt: torch.tensor(ids, device=first_device)
                 for opt, ids in option_tokens_raw.items()}

for opt, ids in option_tokens_raw.items():
    decoded = [tokenizer.decode([i]) for i in ids]
    print(f"  option '{opt}': {decoded}")

# Format texts
conditions = ['B_cult', 'B_unrel']
texts_fmt = {c: format_for_chat(data[c], tokenizer) for c in conditions}

# Register runtime singletons so common helpers can see them
config.model = model
config.tokenizer = tokenizer
config.first_device = first_device

In [ ]:
# Build positions for the edge-KO test
# 3. Format texts + build positions
print("  Building positions...")
texts_fmt = {c: format_for_chat(data[c], tokenizer) for c in conditions}
positions = {c: [] for c in conditions}
n_valid = 0

for c in conditions:
    for i in range(n_total):
        q_text = data[c][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        item = data['items_cult'][i]
        enc = tokenizer(texts_fmt[c][i], return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, q_text, oa, ob, item, tokenizer,
                          item_required=True)
        if sp is None:
            positions[c].append(None)
        else:
            assoc_pos = data['assoc_pos'][i]
            B_tokens = sp['opt_a'] if assoc_pos == 'a' else sp['opt_b']
            A_tokens = sp['opt_b'] if assoc_pos == 'a' else sp['opt_a']
            positions[c].append({
                'item_tokens': sp['item'], 'B_tokens': B_tokens,
                'A_tokens': A_tokens,
            })
            if c == conditions[0]:
                n_valid += 1
        del enc

print(f"  Valid positions: {n_valid}/{n_total}")

In [ ]:
# ================================================================
# QUALITATIVE EXAMPLE — Holi festival on Mistral-7B-Instruct
# Requires: the setup cells above already executed (model loaded, helpers ready,
#           data + positions + option_tokens already built)
# ================================================================
import numpy as np
import torch

print("=" * 80)
print(f"QUALITATIVE EXAMPLE — Holi festival ({CFG['label']})")
print(f"  Heads: {CFG['heads']}")
print("=" * 80)

# 1. Locate the Holi pairs
items_arr = np.array(data['items_cult'])
holi_mask = np.array(['holi' in it.lower() for it in items_arr])
holi_idx = np.where(holi_mask)[0]
print(f"\n  Found {len(holi_idx)} factorial pairs with 'Holi' in item name")
for j, i in enumerate(holi_idx):
    print(f"    [{i}] item='{data['items_cult'][i]}', pos={data['assoc_pos'][i]}, "
          f"R={data['correct_group'][i]}, U={data['third_group'][i]}, "
          f"swap={data['wrong_group'][i]}")

if len(holi_idx) == 0:
    raise RuntimeError("No Holi pairs found — check item names in N4.")

# 2. Build sub-data for these pairs only
sub_conds = ['B_cult', 'B_unrel']
sub_data = {c: [data[c][i] for i in holi_idx] for c in sub_conds}
sub_data['items_cult']    = [data['items_cult'][i] for i in holi_idx]
sub_data['assoc_pos']     = [data['assoc_pos'][i] for i in holi_idx]
sub_data['correct_group'] = [data['correct_group'][i] for i in holi_idx]
sub_data['wrong_group']   = [data['wrong_group'][i] for i in holi_idx]
sub_data['third_group']   = [data['third_group'][i] for i in holi_idx]

sub_texts_fmt = {c: format_for_chat(sub_data[c], tokenizer) for c in sub_conds}
sub_positions = {c: [positions[c][i] for i in holi_idx] for c in sub_conds}

# Sanity check on positions
for c in sub_conds:
    n_valid = sum(1 for p in sub_positions[c] if p is not None)
    print(f"    {c}: {n_valid}/{len(holi_idx)} valid positions")

# 3. Baseline (no KO)
print("\n  [Baseline]")
S_baseline = {}
for c in sub_conds:
    S_baseline[c] = compute_logit_scores_edge(
        model, tokenizer, sub_texts_fmt[c], sub_data[c],
        {}, sub_positions[c], 'B_to_item', option_tokens)
S_match_base = float(np.mean(S_baseline['B_cult']))
S_mism_base  = float(np.mean(S_baseline['B_unrel']))
dS_base      = S_match_base - S_mism_base
abs_dS_base  = abs(dS_base)
print(f"    S_match = {S_match_base:.4f}")
print(f"    S_mism. = {S_mism_base:.4f}")
print(f"    Δ(S) = {dS_base:.4f}, |Δ(S)| = {abs_dS_base:.4f}")

# 4. R→item KO on identified heads
print(f"\n  [R→item KO on {CFG['heads']}]")
S_ko = {}
for c in sub_conds:
    S_ko[c] = compute_logit_scores_edge(
        model, tokenizer, sub_texts_fmt[c], sub_data[c],
        CFG['heads'], sub_positions[c], 'B_to_item', option_tokens)
S_match_ko = float(np.mean(S_ko['B_cult']))
S_mism_ko  = float(np.mean(S_ko['B_unrel']))
dS_ko      = S_match_ko - S_mism_ko
abs_dS_ko  = abs(dS_ko)
shift_match = S_match_ko - S_match_base
shift_mism  = S_mism_ko  - S_mism_base
red_pct = (abs_dS_base - abs_dS_ko) / abs_dS_base * 100
print(f"    S_match = {S_match_ko:.4f}  (shift: {shift_match:+.4f})")
print(f"    S_mism. = {S_mism_ko:.4f}  (shift: {shift_mism:+.4f})")
print(f"    Δ(S) = {dS_ko:.4f}, |Δ(S)| = {abs_dS_ko:.4f}")
print(f"    Item-level reduction in |Δ(S)|: {red_pct:.2f}%")

# 5. Pre-formatted summary for paper
print("\n" + "=" * 80)
print("PAPER-READY SUMMARY")
print("=" * 80)
print(f"  Baseline:   S_match = {S_match_base:.2f}, S_mism. = {S_mism_base:.2f}")
print(f"              |Δ(S)| = {abs_dS_base:.2f}")
print(f"  After KO:   S_match = {S_match_ko:.2f}  (+{shift_match:.2f})")
print(f"              S_mism. = {S_mism_ko:.2f}  ({shift_mism:+.2f})")
print(f"              |Δ(S)| = {abs_dS_ko:.2f}")
print(f"  Item-level reduction: {red_pct:.1f}%")
print(f"  Aggregate average reduction (Table 2): 23.5%")
print()
print("  LaTeX template:")
print(f"    At baseline, $S_\\text{{match}} = {S_match_base:.2f}$ ... "
      f"$S_\\text{{mism.}} = {S_mism_base:.2f}$ ... "
      f"$|\\Delta S| = {abs_dS_base:.2f}$ ... "
      f"$S_\\text{{match}}$ rises to ${S_match_ko:.2f}$ "
      f"(${shift_match:+.2f}$) "
      f"while $S_\\text{{mism.}}$ stays basically unchanged at ${S_mism_ko:.2f}$ "
      f"(${shift_mism:+.2f}$). "
      f"This lowers $|\\Delta S|$ to {abs_dS_ko:.2f}, "
      f"which is a {red_pct:.1f}\\% reduction at the item level "
      f"(larger than the 23.5\\% average across all 66 items).")
print()
print("  Done.")